In [ ]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"

from datasets import load_dataset
import pandas as pd
from convokit import Corpus, download
from pathlib import Path
import random
from tqdm import tqdm

from ssf.Taxonomy import Taxonomy
from ssf.Constants import *
from ssf.ssf_sim import StorySsfSim, CommunitySsfSim
from ssf.ssf_sim.data_utils import (
    build_community_sublabel_counts,
    build_community_varvals_list
)

SBERT_MODEL = "all-MiniLM-L6-v2"
SEED = 50

random.seed(SEED)
taxonomy = Taxonomy(TAXONOMY_DIR)
all_dims = taxonomy.get_dims()
output_dir = Path('outputs/similarity')

/usr2/jmire/ssf/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to /home/jmire/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Load SSF-Corpus

In [2]:
ssf_corpus = load_dataset(SSF_CORPUS_HF, token=True)['full']
ssf_df = ssf_corpus.to_pandas()
reddit_corpus = Corpus(download('reddit-corpus-small'))
ssf_df['text'] = ssf_df['id'].apply(lambda id: reddit_corpus.get_utterance(id).text)

Dataset already exists at /home/jmire/.convokit/saved-corpora/reddit-corpus-small


In [3]:
def sample_pairs(n_items, n_pairs, seed=None):
    """Sample random pairs of indices efficiently."""
    if seed is not None:
        random.seed(seed)
    pairs = set()
    while len(pairs) < n_pairs:
        i, j = random.sample(range(n_items), 2)
        pairs.add((min(i, j), max(i, j)))
    return list(pairs)

### Instance-level SSF-Similarity

In [4]:
num_story_pairs = 1500
story_pairs = sample_pairs(len(ssf_df), num_story_pairs, seed=SEED)

# Initialize calculator with exclusions
story_sim_calc = StorySsfSim(
    taxonomy=taxonomy,
    sbert_model_name=SBERT_MODEL,
    lambda_param=0.667
)

# Compute similarities
similarities = []
for i, j in tqdm(story_pairs):
    row1, row2 = ssf_df.iloc[i], ssf_df.iloc[j]
    sublabels1 = {dim: row1[f"{dim}_labels"] if isinstance(row1[f"{dim}_labels"], list) else [] for dim in all_dims}
    sublabels2 = {dim: row2[f"{dim}_labels"] if isinstance(row2[f"{dim}_labels"], list) else [] for dim in all_dims}
    varvals1 = {dim: taxonomy.get_var_vals(dim, row1[f"{dim}_inference"]) if row1[f"{dim}_inference"] else [] for dim in all_dims}
    varvals2 = {dim: taxonomy.get_var_vals(dim, row2[f"{dim}_inference"]) if row2[f"{dim}_inference"] else [] for dim in all_dims}
    sim = story_sim_calc.compute_similarity(sublabels1, sublabels2, varvals1, varvals2)
    similarities.append({
        'id1': row1['id'],
        'id2': row2['id'],
        'text1': row1['text'],
        'text2': row2['text'],
        'similarity': sim
    })

# Sort by similarity
similarities.sort(key=lambda x: x['similarity'])

# Create DataFrames for top/bottom 10 (for saving)
least_similar_df = pd.DataFrame(similarities[:10])
most_similar_df = pd.DataFrame(similarities[-10:][::-1])

# Save to CSV
setting_dir = output_dir / 'story'
setting_dir.mkdir(parents=True, exist_ok=True)

least_similar_df.to_csv(setting_dir / 'least_similar.csv', index=False)
most_similar_df.to_csv(setting_dir / 'most_similar.csv', index=False)

# Display top 5 most similar
print(f"\n  Top 5 most similar stories:")
display(most_similar_df.head(5))

100%|██████████| 1500/1500 [02:20<00:00, 10.67it/s]


  Top 5 most similar stories:


,id1,id2,text1,text2,similarity
0,e591f6u,e5jqclr,One scar on the thumb looks like a football. ...,My flatmate once served me his home-made 'spag...,0.888419
1,e58vkux,e680ta7,I sliced a decent chunk of flesh off my knuckl...,"When I was 5 and my little brother was 3, he w...",0.866269
2,e5todxy,e5ayzh2,Wow it's amazing how a memory can just get tri...,I've been trying to remember the name of a ser...,0.866014
3,e5op9su,e58ysoi,Halo: Online (The 'unofficial' one) is basical...,I remember a golden gun that was like a shotgu...,0.854779
4,e6wmrkd,e69tg12,They might not just pick people who depate wel...,"I was baptized when i was a child, lived a few...",0.852571


### Community-level SSF-Similarity

In [5]:
print(ssf_df['community'].value_counts().head(10))

community
AskReddit              336
tifu                   269
todayilearned          265
IAmA                   262
books                  253
explainlikeimfive      241
LifeProTips            199
Frugal                 185
MovieDetails           164
relationship_advice    157
Name: count, dtype: int64


In [6]:
communities = ssf_df['community'].unique().tolist()
num_community_pairs = 200
community_pairs = sample_pairs(len(communities), num_community_pairs, seed=SEED)

# Initialize calculator with exclusions
community_sim_calc = CommunitySsfSim(
    taxonomy=taxonomy,
    sbert_model_name=SBERT_MODEL,
    lambda_param=0.667)

# Prepare data (rename columns)
df_prep = ssf_df.copy()
rename_map = {f"{dim}_labels": f"{dim}_cats" for dim in all_dims}
rename_map.update({f"{dim}_inference": f"{dim}_gen" for dim in all_dims})
df_prep = df_prep.rename(columns=rename_map)

sublabel_counts = build_community_sublabel_counts(
    df=df_prep,
    taxonomy=taxonomy,
    groupby_col='community',
    sublabels_to_ignore=['other']
)

varvals_list = build_community_varvals_list(
    df=df_prep,
    taxonomy=taxonomy,
    groupby_col='community'
)

# Compute FULL similarity matrix for ALL communities at once
print(f"  Computing full similarity matrix for {len(communities)} communities...")
sim_matrix = community_sim_calc.compute_similarity(sublabel_counts, varvals_list)

# Extract similarities for sampled pairs
similarities = []
for i, j in tqdm(community_pairs):
    comm1, comm2 = communities[i], communities[j]
    
    # Extract similarity from full matrix
    # Matrix is upper-triangular, so access with sorted community names
    sorted_comms = sorted([comm1, comm2])
    sim_score = sim_matrix.loc[sorted_comms[0], sorted_comms[1]]
    
    # Get story counts
    n_stories_1 = len(ssf_df[ssf_df['community'] == comm1])
    n_stories_2 = len(ssf_df[ssf_df['community'] == comm2])
    
    similarities.append({
        'community1': comm1,
        'community2': comm2,
        'n_stories_1': n_stories_1,
        'n_stories_2': n_stories_2,
        'similarity': sim_score
    })

# Sort by similarity
similarities.sort(key=lambda x: x['similarity'])

# Create DataFrames for top/bottom 10 (for saving)
least_similar_df = pd.DataFrame(similarities[:10])
most_similar_df = pd.DataFrame(similarities[-10:][::-1])

# Save to CSV
setting_dir = output_dir / 'community'
setting_dir.mkdir(parents=True, exist_ok=True)

least_similar_df.to_csv(setting_dir / 'least_similar.csv', index=False)
most_similar_df.to_csv(setting_dir / 'most_similar.csv', index=False)

print(f"  Saved to {setting_dir}")

# Display top 5 most similar
print(f"\n  Top 5 most similar communities:")
display(most_similar_df.head(10))

  Computing full similarity matrix for 55 communities...


100%|██████████| 200/200 [00:00<00:00, 1209.77it/s]

  Saved to outputs/similarity/community

  Top 5 most similar communities:


,community1,community2,n_stories_1,n_stories_2,similarity
0,IAmA,science,262,135,0.989455
1,tifu,AskReddit,269,336,0.959585
2,nfl,baseball,75,36,0.957334
3,Games,gaming,103,116,0.955758
4,MMA,baseball,87,36,0.948632
5,australia,technology,46,110,0.944340
6,science,LifeProTips,135,199,0.933552
7,books,gaming,253,116,0.925989
8,australia,todayilearned,46,265,0.922984
9,hockey,MMA,73,87,0.919219
